In [0]:
# Use the correct catalog and schema
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA ecommerce_lakehouse")

bronze_table = "bronze_clickstream_events"
error_table = "bronze_error_records"

source_path = "/Volumes/workspace/ecommerce_lakehouse/raw_events_volume/stream_input"
checkpoint_path = "/Volumes/workspace/ecommerce_lakehouse/raw_events_volume/checkpoints/bronze"
error_checkpoint_path = "/Volumes/workspace/ecommerce_lakehouse/raw_events_volume/checkpoints/error_records"
schema_path = "/Volumes/workspace/ecommerce_lakehouse/raw_events_volume/schema_tracking"

print("Project context initialized")

In [0]:
# Define Strong Schema. Production pipelines always enforce schema.This prevents unexpected data corruption.
from pyspark.sql.types import *

event_schema = StructType([
    StructField("user_id", StringType()),
    StructField("session_id", StringType()),
    StructField("product_id", StringType()),
    StructField("event_type", StringType()),
    StructField("price", DoubleType()),
    StructField("device", StringType()),
    StructField("country", StringType()),
    StructField("fraud_flag", IntegerType()),
    StructField("ts", LongType())
])

In [0]:
# Configure Auto Loader (Schema Evolution Enabled),Auto Loader will automatically track schema changes.If a new field appears in future data, Auto Loader automatically updates the schema.This enables: incremental ingestion; schema tracking; Auto Loader scalability ; ⚠️ Important: we don't include schema evolution because it conflicts with manual schema.

stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", schema_path)
    .option("cloudFiles.inferColumnTypes", "true")
    .schema(event_schema)
    .load(source_path)
)

In [0]:
#Data Quality Split.This creates valid vs invalid streams.This prevents bad records from corrupting Bronze tables.
from pyspark.sql.functions import col

valid_df = (
    stream_df
    .filter(col("user_id").isNotNull())
    .filter(col("product_id").isNotNull())
    .filter(col("price") > 0)
)

invalid_df = (
    stream_df
    .filter(
        (col("user_id").isNull()) |
        (col("product_id").isNull()) |
        (col("price") <= 0)
    )
)

In [0]:
#Add Ingestion Metadata Columns ,Now every row contains:event timestamp,ingestion timestamp . Which allows pipeline latency tracking.

from pyspark.sql.functions import current_timestamp

bronze_df = (
    valid_df
    .withColumn("ingest_time", current_timestamp())
)

In [0]:
# Write Valid Records to Bronze Table ⚠️ We use "availableNow trigger" because community edition databricks cluster doesn't support infinite streaming.This ingests all available data and stops automatically.
bronze_query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("workspace.ecommerce_lakehouse.bronze_clickstream_events")
)


In [0]:
# Write Invalid Records to Quarantine Table.This table stores bad records for debugging.
error_query = (
    invalid_df
    .withColumn("error_time", current_timestamp())
    .writeStream
    .format("delta")
    .option("checkpointLocation", error_checkpoint_path)
    .outputMode("append")
    .trigger(availableNow=True)
    .toTable("workspace.ecommerce_lakehouse.bronze_error_records")
)

In [0]:
# Verify Bronze Data
spark.sql("""
SELECT event_type, COUNT(*)
FROM workspace.ecommerce_lakehouse.bronze_clickstream_events
GROUP BY event_type
""").show()

In [0]:
#Create Pipeline Metrics Table
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.ecommerce_lakehouse.pipeline_metrics
(
    metric_time TIMESTAMP,
    total_events BIGINT
)
""")

In [0]:
# Insert Pipeline Metrics.This enables pipeline monitoring dashboards later
from pyspark.sql.functions import current_timestamp, count

metrics_df = (
    spark.read.table("workspace.ecommerce_lakehouse.bronze_clickstream_events")
    .agg(count("*").alias("total_events"))
    .withColumn("metric_time", current_timestamp())
)

metrics_df.write.mode("append").saveAsTable(
    "workspace.ecommerce_lakehouse.pipeline_metrics"
)

In [0]:
#Delta Optimization.This improves query performance.
spark.sql("""
OPTIMIZE workspace.ecommerce_lakehouse.bronze_clickstream_events
ZORDER BY (user_id)
""")

In [0]:
# Inspect Delta Table History
spark.sql("""
DESCRIBE HISTORY workspace.ecommerce_lakehouse.bronze_clickstream_events
""").display()

In [0]:
#Verify Bronze Table
spark.read.table("workspace.ecommerce_lakehouse.bronze_clickstream_events").display()


In [0]:
#Confirm Data Loaded
spark.sql("""
SELECT COUNT(*)
FROM workspace.ecommerce_lakehouse.bronze_clickstream_events
""").show()